# Research Paper Retrieval System with Semantic + Recency Weighting

This notebook implements a RAG system that retrieves research papers based on semantic similarity and publication recency.

In [1]:
!pip install -q sentence-transformers faiss-cpu pandas numpy scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
import re
from typing import List, Dict, Tuple
import json
from datetime import datetime

print("✓ All imports completed successfully")

o:\Programming\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ All imports completed successfully


In [3]:
class PaperPreprocessor:
    """Loads, preprocesses, and tokenizes research papers."""
    
    def __init__(self):
        self.papers = []
        self.metadata = []
    
    def load_papers(self, data_source: List[Dict]) -> pd.DataFrame:
        """
        Load research papers from a data source.
        
        Args:
            data_source: List of dicts with keys: 'title', 'abstract', 'year', 'authors', 'id'
        
        Returns:
            DataFrame with papers and metadata
        """
        self.papers = data_source
        df = pd.DataFrame(data_source)
        print(f"✓ Loaded {len(df)} research papers")
        return df
    
    def preprocess_text(self, text: str) -> str:
        """
        Preprocess text: lowercase, remove special chars, normalize whitespace.
        
        Args:
            text: Raw text to preprocess
        
        Returns:
            Cleaned text
        """
        # Lowercase
        text = text.lower()
        # Remove URLs
        text = re.sub(r'http\S+|www\S+', '', text)
        # Remove special characters but keep basic punctuation
        text = re.sub(r'[^a-zA-Z0-9\s\.\,\-]', '', text)
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    def generate_document(self, paper: Dict, fields: List[str] = None) -> str:
        """
        Generate document from structured paper information.
        Selects relevant fields to form document for embeddings.
        
        Args:
            paper: Paper data dict
            fields: Fields to include. Default: ['title', 'abstract', 'authors']
        
        Returns:
            Concatenated document string
        """
        if fields is None:
            fields = ['title', 'abstract', 'authors']
        
        document_parts = []
        for field in fields:
            if field in paper and paper[field]:
                value = str(paper[field])
                document_parts.append(value)
        
        document = " ".join(document_parts)
        return self.preprocess_text(document)

# Test the preprocessor
preprocessor = PaperPreprocessor()
print("✓ PaperPreprocessor initialized")

✓ PaperPreprocessor initialized


In [4]:
class EmbeddingGenerator:
    """Generates embeddings to encode semantic meaning."""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize embedding generator with a pre-trained model.
        
        Args:
            model_name: Name of sentence-transformer model
        """
        print(f"Loading embedding model: {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()
        print(f"✓ Model loaded. Embedding dimension: {self.embedding_dim}")
    
    def generate_embeddings(self, documents: List[str], batch_size: int = 32) -> np.ndarray:
        """
        Generate embeddings for a list of documents.
        
        Args:
            documents: List of text documents
            batch_size: Batch size for processing
        
        Returns:
            Array of embeddings (N, embedding_dim)
        """
        print(f"Generating embeddings for {len(documents)} documents...")
        embeddings = self.model.encode(
            documents,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        return embeddings.astype('float32')

# Test the embedding generator
embedding_gen = EmbeddingGenerator()
print("✓ EmbeddingGenerator initialized")

Loading embedding model: all-MiniLM-L6-v2...


o:\Programming\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\shanu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2932.04it/s]
BertModel LOAD REPORT from: sentence-tra

✓ Model loaded. Embedding dimension: 384
✓ EmbeddingGenerator initialized


In [5]:
class FAISSIndexManager:
    """Stores and manages embeddings in FAISS for efficient similarity search."""
    
    def __init__(self, embedding_dim: int):
        """
        Initialize FAISS index.
        
        Args:
            embedding_dim: Dimensionality of embeddings
        """
        # Use IndexFlatL2 for exact search (can switch to IVF for larger datasets)
        self.index = faiss.IndexFlatL2(embedding_dim)
        self.id_map = []  # Map FAISS internal ids to paper ids
        print(f"✓ FAISS index initialized with dimension {embedding_dim}")
    
    def add_embeddings(self, embeddings: np.ndarray, paper_ids: List[str]) -> None:
        """
        Add embeddings to the FAISS index.
        
        Args:
            embeddings: Array of shape (N, embedding_dim)
            paper_ids: List of paper identifiers
        """
        self.index.add(embeddings)
        self.id_map.extend(paper_ids)
        print(f"✓ Added {len(embeddings)} embeddings to FAISS index")
    
    def search_nearest_neighbors(self, query_embedding: np.ndarray, k: int = 5) -> Tuple[np.ndarray, List[str]]:
        """
        Search for k nearest neighbors.
        
        Args:
            query_embedding: Query embedding (1, embedding_dim)
            k: Number of neighbors to retrieve
        
        Returns:
            Tuple of (distances, paper_ids)
        """
        distances, indices = self.index.search(query_embedding, k)
        paper_ids = [self.id_map[idx] for idx in indices[0]]
        return distances[0], paper_ids

# Test the FAISS manager
faiss_manager = FAISSIndexManager(embedding_dim=embedding_gen.embedding_dim)
print("✓ FAISSIndexManager initialized")

✓ FAISS index initialized with dimension 384
✓ FAISSIndexManager initialized


In [6]:
class ResearchPaperRetriever:
    """
    Main retriever class combining semantic similarity and recency weighting.
    
    Hybrid Retrieval Strategy:
    - Semantic similarity: L2 distance from embedding similarity
    - Recency: Normalized publication year (more recent = higher score)
    - Hybrid Score: (1 - alpha) * semantic_score + alpha * recency_score
    """
    
    def __init__(self, preprocessor, embedding_gen, faiss_manager, alpha: float = 0.3):
        """
        Initialize the retriever.
        
        Args:
            preprocessor: PaperPreprocessor instance
            embedding_gen: EmbeddingGenerator instance
            faiss_manager: FAISSIndexManager instance
            alpha: Weight for recency (0-1). Higher = more weight on recency
        """
        self.preprocessor = preprocessor
        self.embedding_gen = embedding_gen
        self.faiss_manager = faiss_manager
        self.alpha = alpha  # Recency weight
        self.papers_df = None
        self.scaler = MinMaxScaler()
        
    def build_index(self, papers: List[Dict]) -> None:
        """
        Build the complete index from paper list.
        
        Args:
            papers: List of paper dicts with keys: title, abstract, year, authors, id
        """
        # Load papers
        self.papers_df = self.preprocessor.load_papers(papers)
        
        # Generate documents from structured information
        documents = [
            self.preprocessor.generate_document(paper)
            for paper in papers
        ]
        
        # Generate embeddings
        embeddings = self.embedding_gen.generate_embeddings(documents)
        
        # Add to FAISS
        paper_ids = [str(p.get('id', i)) for i, p in enumerate(papers)]
        self.faiss_manager.add_embeddings(embeddings, paper_ids)
        
        # Normalize years for recency scoring
        years = self.papers_df['year'].values.reshape(-1, 1)
        self.papers_df['year_normalized'] = self.scaler.fit_transform(years).flatten()
        
        print(f"\n✓ Index built successfully with {len(papers)} papers")
    
    def _calculate_recency_score(self, years: np.ndarray) -> np.ndarray:
        """
        Calculate recency score from normalized years.
        More recent = higher score.
        
        Args:
            years: Array of publication years
        
        Returns:
            Normalized recency scores (0-1)
        """
        years_reshaped = years.reshape(-1, 1)
        normalized = self.scaler.transform(years_reshaped).flatten()
        return normalized
    
    def _calculate_semantic_score(self, distances: np.ndarray) -> np.ndarray:
        """
        Convert L2 distances to similarity scores (0-1).
        Lower distance = higher similarity.
        
        Args:
            distances: L2 distances from query
        
        Returns:
            Similarity scores (0-1)
        """
        # Normalize distances to 0-1 range (lower distance = higher score)
        max_dist = np.max(distances) if np.max(distances) > 0 else 1.0
        semantic_scores = 1 - (distances / (max_dist + 1e-10))
        return semantic_scores
    
    def retrieve_with_recency(self, query: str, k: int = 5) -> List[Dict]:
        """
        Retrieve top-k research papers based on semantic similarity AND recency.
        
        Challenge Solution:
        1. Calculate semantic similarity score (L2 distance normalization)
        2. Calculate recency score (year normalization)
        3. Combine: hybrid_score = (1 - alpha) * semantic + alpha * recency
        4. Sort by recency within semantically similar results
        5. Return exactly k documents ordered by recency
        
        Args:
            query: Query string
            k: Number of papers to retrieve
        
        Returns:
            List of k papers sorted by recency (most recent first)
        """
        # Preprocess and embed query
        query_doc = self.preprocessor.preprocess_text(query)
        query_embedding = self.embedding_gen.generate_embeddings([query_doc])
        
        # Search more candidates than needed to ensure we get good results after filtering
        search_k = min(k * 3, len(self.papers_df))  # Search 3x k candidates
        distances, paper_ids = self.faiss_manager.search_nearest_neighbors(query_embedding, search_k)
        
        # Get paper indices
        paper_indices = [
            self.papers_df[self.papers_df['id'].astype(str) == pid].index[0]
            if len(self.papers_df[self.papers_df['id'].astype(str) == pid]) > 0
            else None
            for pid in paper_ids
        ]
        paper_indices = [idx for idx in paper_indices if idx is not None]
        
        # Calculate scores
        semantic_scores = self._calculate_semantic_score(distances[:len(paper_indices)])
        years_for_results = self.papers_df.loc[paper_indices, 'year'].values
        recency_scores = self._calculate_recency_score(years_for_results)
        
        # Hybrid score: (1-alpha)*semantic + alpha*recency
        hybrid_scores = (1 - self.alpha) * semantic_scores + self.alpha * recency_scores
        
        # Create result list with scores
        results = []
        for idx, paper_idx in enumerate(paper_indices):
            paper = self.papers_df.loc[paper_idx].to_dict()
            paper['semantic_score'] = float(semantic_scores[idx])
            paper['recency_score'] = float(recency_scores[idx])
            paper['hybrid_score'] = float(hybrid_scores[idx])
            results.append(paper)
        
        # Sort by hybrid score, then by recency (most recent first)
        results.sort(key=lambda x: (-x['hybrid_score'], -x['year']))
        
        # Return exactly k results, sorted by recency
        results = results[:k]
        results.sort(key=lambda x: -x['year'])  # Sort by recency (most recent first)
        
        print(f"\n✓ Retrieved {len(results)} research papers")
        print(f"  (alpha={self.alpha}: {(1-self.alpha)*100:.0f}% semantic, {self.alpha*100:.0f}% recency)")
        
        return results

# Initialize retriever
retriever = ResearchPaperRetriever(
    preprocessor=preprocessor,
    embedding_gen=embedding_gen,
    faiss_manager=faiss_manager,
    alpha=0.3  # 70% semantic, 30% recency
)
print("✓ ResearchPaperRetriever initialized")

✓ ResearchPaperRetriever initialized


## Sample Data: Research Papers on Machine Learning

In [7]:
# Sample dataset of research papers
sample_papers = [
    {
        'id': 1,
        'title': 'Attention Is All You Need',
        'abstract': 'The dominant sequence transduction models are based on complex recurrent or convolutional neural networks. We propose a new simple network architecture based on attention mechanisms. The model relies entirely on self-attention to compute representations.',
        'authors': 'Vaswani et al.',
        'year': 2017
    },
    {
        'id': 2,
        'title': 'BERT: Pre-training of Deep Bidirectional Transformers',
        'abstract': 'We introduce BERT, a new method of pre-training language representations. Unlike recent language representation models, BERT is designed to pre-train deep bidirectional representations by jointly conditioning on both left and right context.',
        'authors': 'Devlin et al.',
        'year': 2018
    },
    {
        'id': 3,
        'title': 'Language Models are Unsupervised Multitask Learners',
        'abstract': 'GPT-2 is a large-scale unsupervised language model that achieves state-of-the-art performance on language modeling and zero-shot task transfer. We demonstrate that language models begin learning these tasks without any explicit supervision when trained on a large enough corpus.',
        'authors': 'Radford et al.',
        'year': 2019
    },
    {
        'id': 4,
        'title': 'Convolutional Neural Networks for Visual Recognition',
        'abstract': 'We present a comprehensive study of convolutional neural networks for image classification. We explore various architectures and training strategies that achieve state-of-the-art results on benchmark datasets.',
        'authors': 'Krizhevsky et al.',
        'year': 2012
    },
    {
        'id': 5,
        'title': 'Efficient Estimation of Word Representations in Vector Space',
        'abstract': 'We propose two novel model architectures for learning word representations from very large datasets. Word2Vec introduces methods to generate high-quality word embeddings efficiently using neural networks.',
        'authors': 'Mikolov et al.',
        'year': 2013
    },
    {
        'id': 6,
        'title': 'GloVe: Global Vectors for Word Representation',
        'abstract': 'We present a new model for learning word representations. The model efficiently leverages both global matrix factorization and local context window methods. It produces a vector space with meaningful substructure.',
        'authors': 'Pennington et al.',
        'year': 2014
    },
    {
        'id': 7,
        'title': 'Vision Transformer: An Image is Worth 16x16 Words',
        'abstract': 'While the Transformer architecture has become the de-facto standard for natural language processing, its applications to computer vision remain limited. We show that a pure transformer applied directly to sequences of image patches can perform very well on image classification tasks.',
        'authors': 'Dosovitskiy et al.',
        'year': 2020
    },
    {
        'id': 8,
        'title': 'Deep Residual Learning for Image Recognition',
        'abstract': 'Deep neural networks are difficult to train. We present a residual learning framework to ease the training of networks that are substantially deeper than those used previously. We explicitly reformulate the layers as learning residual functions.',
        'authors': 'He et al.',
        'year': 2015
    },
    {
        'id': 9,
        'title': 'Generative Adversarial Nets',
        'abstract': 'We propose a new framework for estimating generative models via an adversarial process, in which we simultaneously train two models: a generative model G and a discriminative model D.',
        'authors': 'Goodfellow et al.',
        'year': 2014
    },
    {
        'id': 10,
        'title': 'Neural Machine Translation by Jointly Learning to Align and Translate',
        'abstract': 'Neural machine translation has emerged as a promising approach for automated translation. However, standard models often struggle with long sequences. We propose an extension to the attention mechanism that allows the model to search for relevant source words dynamically.',
        'authors': 'Bahdanau et al.',
        'year': 2015
    },
    {
        'id': 11,
        'title': 'Learning Transferable Visual Models From Natural Language Supervision',
        'abstract': 'Cross-modal pre-training is a powerful mechanism for learning visual features. CLIP learns an image encoder and text encoder jointly via contrastive learning to predict which images and text descriptions go together.',
        'authors': 'Radford et al.',
        'year': 2021
    },
    {
        'id': 12,
        'title': 'Diffusion Models Beat GANs on Image Synthesis',
        'abstract': 'Diffusion models have recently shown great promise for generative modeling, particularly when combined with guidance methods. We present a method for training large-scale diffusion models that beats GANs on image synthesis benchmarks.',
        'authors': 'Dhariwal et al.',
        'year': 2021
    },
]

print(f"✓ Created sample dataset with {len(sample_papers)} research papers")
print(f"  Year range: {min(p['year'] for p in sample_papers)}-{max(p['year'] for p in sample_papers)}")

✓ Created sample dataset with 12 research papers
  Year range: 2012-2021


In [8]:
# Build the index
retriever.build_index(sample_papers)

✓ Loaded 12 research papers
Generating embeddings for 12 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.28it/s]

✓ Added 12 embeddings to FAISS index

✓ Index built successfully with 12 papers


In [9]:
def display_results(results: List[Dict], query: str) -> None:
    """Display retrieval results in a formatted table."""
    print(f"\n{'='*100}")
    print(f"Query: {query}")
    print(f"{'='*100}\n")
    
    for i, paper in enumerate(results, 1):
        print(f"{i}. [{paper['year']}] {paper['title']}")
        print(f"   Authors: {paper['authors']}")
        print(f"   Semantic Score: {paper['semantic_score']:.4f} | Recency Score: {paper['recency_score']:.4f} | Hybrid Score: {paper['hybrid_score']:.4f}")
        print(f"   Abstract: {paper['abstract'][:150]}...")
        print()

# Test Query 1: Deep Learning
query1 = "deep learning neural networks"
results1 = retriever.retrieve_with_recency(query1, k=5)
display_results(results1, query1)

Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 49.34it/s]


✓ Retrieved 5 research papers
  (alpha=0.3: 70% semantic, 30% recency)

Query: deep learning neural networks

1. [2021] Learning Transferable Visual Models From Natural Language Supervision
   Authors: Radford et al.
   Semantic Score: 0.0748 | Recency Score: 1.0000 | Hybrid Score: 0.3524
   Abstract: Cross-modal pre-training is a powerful mechanism for learning visual features. CLIP learns an image encoder and text encoder jointly via contrastive l...

2. [2021] Diffusion Models Beat GANs on Image Synthesis
   Authors: Dhariwal et al.
   Semantic Score: 0.0191 | Recency Score: 1.0000 | Hybrid Score: 0.3134
   Abstract: Diffusion models have recently shown great promise for generative modeling, particularly when combined with guidance methods. We present a method for ...

3. [2020] Vision Transformer: An Image is Worth 16x16 Words
   Authors: Dosovitskiy et al.
   Semantic Score: 0.1386 | Recency Score: 0.8889 | Hybrid Score: 0.3637
   Abstract: While the Transformer architecture has 

In [10]:
# Test Query 2: Transformers
query2 = "transformers attention mechanism"
results2 = retriever.retrieve_with_recency(query2, k=5)
display_results(results2, query2)

Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.73it/s]


✓ Retrieved 5 research papers
  (alpha=0.3: 70% semantic, 30% recency)

Query: transformers attention mechanism

1. [2021] Learning Transferable Visual Models From Natural Language Supervision
   Authors: Radford et al.
   Semantic Score: 0.2433 | Recency Score: 1.0000 | Hybrid Score: 0.4703
   Abstract: Cross-modal pre-training is a powerful mechanism for learning visual features. CLIP learns an image encoder and text encoder jointly via contrastive l...

2. [2021] Diffusion Models Beat GANs on Image Synthesis
   Authors: Dhariwal et al.
   Semantic Score: 0.1250 | Recency Score: 1.0000 | Hybrid Score: 0.3875
   Abstract: Diffusion models have recently shown great promise for generative modeling, particularly when combined with guidance methods. We present a method for ...

3. [2020] Vision Transformer: An Image is Worth 16x16 Words
   Authors: Dosovitskiy et al.
   Semantic Score: 0.3048 | Recency Score: 0.8889 | Hybrid Score: 0.4800
   Abstract: While the Transformer architecture h

## Testing Recency Weighting Trade-offs

Compare results with different alpha values to show the impact of recency weighting.

In [11]:
# Test with different alpha values
test_query = "word embeddings representation learning"

print("\n" + "="*100)
print("IMPACT OF RECENCY WEIGHTING")
print("="*100)

for alpha in [0.0, 0.3, 0.6, 1.0]:
    retriever.alpha = alpha
    results = retriever.retrieve_with_recency(test_query, k=5)
    print(f"\nAlpha = {alpha} ({int((1-alpha)*100)}% semantic, {int(alpha*100)}% recency):")
    print("-" * 100)
    for i, paper in enumerate(results, 1):
        print(f"  {i}. [{paper['year']}] {paper['title'][:60]}")
        print(f"     Semantic: {paper['semantic_score']:.4f} | Recency: {paper['recency_score']:.4f} | Hybrid: {paper['hybrid_score']:.4f}")


IMPACT OF RECENCY WEIGHTING
Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 38.94it/s]



✓ Retrieved 5 research papers
  (alpha=0.0: 100% semantic, 0% recency)

Alpha = 0.0 (100% semantic, 0% recency):
----------------------------------------------------------------------------------------------------
  1. [2021] Learning Transferable Visual Models From Natural Language Su
     Semantic: 0.2323 | Recency: 1.0000 | Hybrid: 0.2323
  2. [2019] Language Models are Unsupervised Multitask Learners
     Semantic: 0.2228 | Recency: 0.7778 | Hybrid: 0.2228
  3. [2018] BERT: Pre-training of Deep Bidirectional Transformers
     Semantic: 0.2985 | Recency: 0.6667 | Hybrid: 0.2985
  4. [2014] GloVe: Global Vectors for Word Representation
     Semantic: 0.4765 | Recency: 0.2222 | Hybrid: 0.4765
  5. [2013] Efficient Estimation of Word Representations in Vector Space
     Semantic: 0.6532 | Recency: 0.1111 | Hybrid: 0.6532
Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.46it/s]



✓ Retrieved 5 research papers
  (alpha=0.3: 70% semantic, 30% recency)

Alpha = 0.3 (70% semantic, 30% recency):
----------------------------------------------------------------------------------------------------
  1. [2021] Learning Transferable Visual Models From Natural Language Su
     Semantic: 0.2323 | Recency: 1.0000 | Hybrid: 0.4626
  2. [2019] Language Models are Unsupervised Multitask Learners
     Semantic: 0.2228 | Recency: 0.7778 | Hybrid: 0.3893
  3. [2018] BERT: Pre-training of Deep Bidirectional Transformers
     Semantic: 0.2985 | Recency: 0.6667 | Hybrid: 0.4090
  4. [2014] GloVe: Global Vectors for Word Representation
     Semantic: 0.4765 | Recency: 0.2222 | Hybrid: 0.4002
  5. [2013] Efficient Estimation of Word Representations in Vector Space
     Semantic: 0.6532 | Recency: 0.1111 | Hybrid: 0.4906
Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 71.58it/s]


✓ Retrieved 5 research papers


  (alpha=0.6: 40% semantic, 60% recency)

Alpha = 0.6 (40% semantic, 60% recency):
----------------------------------------------------------------------------------------------------
  1. [2021] Learning Transferable Visual Models From Natural Language Su
     Semantic: 0.2323 | Recency: 1.0000 | Hybrid: 0.6929
  2. [2021] Diffusion Models Beat GANs on Image Synthesis
     Semantic: 0.0000 | Recency: 1.0000 | Hybrid: 0.6000
  3. [2020] Vision Transformer: An Image is Worth 16x16 Words
     Semantic: 0.1393 | Recency: 0.8889 | Hybrid: 0.5890
  4. [2019] Language Models are Unsupervised Multitask Learners
     Semantic: 0.2228 | Recency: 0.7778 | Hybrid: 0.5558
  5. [2018] BERT: Pre-training of Deep Bidirectional Transformers
     Semantic: 0.2985 | Recency: 0.6667 | Hybrid: 0.5194
Generating embeddings for 1 documents...


Batches: 100%|██████████| 1/1 [00:00<00:00, 64.67it/s]


✓ Retrieved 5 research papers
  (alpha=1.0: 0% semantic, 100% recency)

Alpha = 1.0 (0% semantic, 100% recency):
----------------------------------------------------------------------------------------------------
  1. [2021] Learning Transferable Visual Models From Natural Language Su
     Semantic: 0.2323 | Recency: 1.0000 | Hybrid: 1.0000
  2. [2021] Diffusion Models Beat GANs on Image Synthesis
     Semantic: 0.0000 | Recency: 1.0000 | Hybrid: 1.0000
  3. [2020] Vision Transformer: An Image is Worth 16x16 Words
     Semantic: 0.1393 | Recency: 0.8889 | Hybrid: 0.8889
  4. [2019] Language Models are Unsupervised Multitask Learners
     Semantic: 0.2228 | Recency: 0.7778 | Hybrid: 0.7778
  5. [2018] BERT: Pre-training of Deep Bidirectional Transformers
     Semantic: 0.2985 | Recency: 0.6667 | Hybrid: 0.6667


## Complete Usage Guide

### Step 1: Prepare Your Data
Papers should have: `id`, `title`, `abstract`, `authors`, `year`

### Step 2: Initialize Components
```python
preprocessor = PaperPreprocessor()
embedding_gen = EmbeddingGenerator(model_name="all-MiniLM-L6-v2")
faiss_manager = FAISSIndexManager(embedding_dim=embedding_gen.embedding_dim)
retriever = ResearchPaperRetriever(preprocessor, embedding_gen, faiss_manager, alpha=0.3)
```

### Step 3: Build Index
```python
retriever.build_index(papers)
```

### Step 4: Retrieve Papers
```python
results = retriever.retrieve_with_recency(query="your query here", k=5)
```

### Key Features:
- **Semantic Similarity**: Uses sentence-transformers for embedding generation
- **Recency Scoring**: Normalized publication year (more recent = higher)
- **Hybrid Ranking**: Combined semantic + recency scores
- **Exact K Results**: Always returns exactly k documents
- **Sorted by Recency**: Results ordered by publication year (most recent first)
- **Configurable Alpha**: `alpha` parameter controls semantic vs. recency tradeoff

### Hybrid Score Formula:
```
hybrid_score = (1 - α) × semantic_score + α × recency_score
```
Where α ∈ [0, 1]
- α = 0: Pure semantic similarity
- α = 0.3: 70% semantic, 30% recency (recommended)
- α = 1.0: Pure recency

## Advanced Usage: Custom Dataset

You can easily use your own research papers dataset.

In [12]:
# Example: Load from CSV
example_code = """
import pandas as pd

# Load papers from CSV
df = pd.read_csv('papers.csv')
papers = df[['id', 'title', 'abstract', 'authors', 'year']].to_dict('records')

# Create retriever
preprocessor = PaperPreprocessor()
embedding_gen = EmbeddingGenerator()
faiss_manager = FAISSIndexManager(embedding_gen.embedding_dim)
retriever = ResearchPaperRetriever(preprocessor, embedding_gen, faiss_manager, alpha=0.35)

# Build and query
retriever.build_index(papers)
results = retriever.retrieve_with_recency("your query", k=10)

# Results are ordered by recency (most recent first)
for paper in results:
    print(f"{paper['year']}: {paper['title']}")
"""

print("Example code for loading custom dataset:")
print(example_code)

Example code for loading custom dataset:

import pandas as pd

# Load papers from CSV
df = pd.read_csv('papers.csv')
papers = df[['id', 'title', 'abstract', 'authors', 'year']].to_dict('records')

# Create retriever
preprocessor = PaperPreprocessor()
embedding_gen = EmbeddingGenerator()
faiss_manager = FAISSIndexManager(embedding_gen.embedding_dim)
retriever = ResearchPaperRetriever(preprocessor, embedding_gen, faiss_manager, alpha=0.35)

# Build and query
retriever.build_index(papers)
results = retriever.retrieve_with_recency("your query", k=10)

# Results are ordered by recency (most recent first)
for paper in results:
    print(f"{paper['year']}: {paper['title']}")



## How to Add Your Own Research Papers

Your papers must follow this format with these required fields:

| Field | Type | Description |
|-------|------|-------------|
| `id` | int | Unique identifier |
| `title` | str | Paper title |
| `abstract` | str | Paper abstract/description |
| `authors` | str | Authors (comma-separated) |
| `year` | int | Publication year |

In [ ]:
# OPTION 1: Load from CSV file
print("="*100)
print("OPTION 1: Load from CSV")
print("="*100)
print("""
# Create a CSV file with columns: id, title, abstract, authors, year
# Example format:
# id,title,abstract,authors,year
# 1,"Paper Title 1","Abstract content here","Author1, Author2",2023
# 2,"Paper Title 2","Abstract content here","Author3, Author4",2024

import pandas as pd

# Read CSV file
df = pd.read_csv('your_papers.csv')

# Ensure all required columns exist
required_cols = ['id', 'title', 'abstract', 'authors', 'year']
if all(col in df.columns for col in required_cols):
    papers = df[required_cols].to_dict('records')
    print(f"✓ Loaded {len(papers)} papers from CSV")
else:
    print("✗ CSV missing required columns:", required_cols)
""")

In [ ]:
# OPTION 2: Load from JSON file
print("\n" + "="*100)
print("OPTION 2: Load from JSON")
print("="*100)
print("""
import json

# JSON file format:
# [
#   {
#     "id": 1,
#     "title": "Paper Title",
#     "abstract": "Abstract text...",
#     "authors": "Author1, Author2",
#     "year": 2023
#   },
#   {
#     "id": 2,
#     "title": "Another Paper",
#     "abstract": "Abstract text...",
#     "authors": "Author3",
#     "year": 2024
#   }
# ]

with open('your_papers.json', 'r') as f:
    papers = json.load(f)
    print(f"✓ Loaded {len(papers)} papers from JSON")
""")

In [ ]:
# OPTION 3: Direct Python list (copy-paste from your data)
print("\n" + "="*100)
print("OPTION 3: Direct Python List")
print("="*100)
print("""
# Paste your papers directly as a Python list:

papers = [
    {
        'id': 1,
        'title': 'Your Paper Title',
        'abstract': 'Your paper abstract goes here...',
        'authors': 'You, CoAuthor',
        'year': 2024
    },
    {
        'id': 2,
        'title': 'Another Paper',
        'abstract': 'Its abstract...',
        'authors': 'Author Name',
        'year': 2024
    },
    # Add more papers...
]

print(f"✓ Loaded {len(papers)} papers")
""")

## Quick Start: Run with Your Papers

Follow these 3 simple steps to use your own papers.

In [3]:
# STEP 1: Prepare your papers
# Replace this with your own papers list, CSV load, or JSON load

my_papers = [
    {
        'id': 1,
        'title': 'Neural Networks in Healthcare',
        'abstract': 'This paper explores the application of neural networks for medical diagnosis and patient monitoring systems.',
        'authors': 'Smith, Johnson, Williams',
        'year': 2024
    },
    {
        'id': 2,
        'title': 'Efficient Language Models',
        'abstract': 'We present optimized techniques for training large language models with reduced computational requirements.',
        'authors': 'Chen, Lee, Park',
        'year': 2024
    },
    {
        'id': 3,
        'title': 'Quantum Computing Basics',
        'abstract': 'An introduction to quantum algorithms and their applications in solving classical problems.',
        'authors': 'Kumar, Shah, Patel',
        'year': 2023
    },
]

print(f"✓ Prepared {len(my_papers)} papers")
print(f"  Papers: {[p['title'][:40] for p in my_papers]}")

✓ Prepared 3 papers
  Papers: ['Neural Networks in Healthcare', 'Efficient Language Models', 'Quantum Computing Basics']


In [4]:
# STEP 2: Create a retrieval system
from sentence_transformers import SentenceTransformer
import numpy as np

# Initialize the embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create embeddings for all papers
paper_embeddings = []
paper_texts = []

for paper in my_papers:
    # Combine title and abstract for better retrieval
    text = f"{paper['title']} {paper['abstract']}"
    paper_texts.append(text)
    embedding = embedding_model.encode(text)
    paper_embeddings.append(embedding)

paper_embeddings = np.array(paper_embeddings)

print(f"✓ Created embeddings for {len(my_papers)} papers")
print(f"  Embedding dimension: {paper_embeddings.shape[1]}")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6978.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Created embeddings for 3 papers
  Embedding dimension: 384


In [5]:
# STEP 3: Query and retrieve papers
def search_papers(query, top_k=2):
    """Search for papers similar to the query"""
    # Encode the query
    query_embedding = embedding_model.encode(query)
    
    # Calculate similarities
    similarities = np.dot(paper_embeddings, query_embedding) / (
        np.linalg.norm(paper_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    
    # Get top-k results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'paper': my_papers[idx],
            'similarity': float(similarities[idx])
        })
    
    return results

# Example searches
queries = [
    "machine learning in healthcare",
    "optimization of neural networks",
    "quantum algorithms"
]

print("✓ Completed the paper retrieval pipeline!")
print("\nExample search results:\n")

for query in queries:
    print(f"Query: '{query}'")
    results = search_papers(query, top_k=2)
    for i, result in enumerate(results, 1):
        print(f"  {i}. {result['paper']['title']} (similarity: {result['similarity']:.3f})")
    print()


✓ Completed the paper retrieval pipeline!

Example search results:

Query: 'machine learning in healthcare'
  1. Neural Networks in Healthcare (similarity: 0.606)
  2. Efficient Language Models (similarity: 0.219)

Query: 'optimization of neural networks'
  1. Neural Networks in Healthcare (similarity: 0.535)
  2. Efficient Language Models (similarity: 0.299)

Query: 'quantum algorithms'
  1. Quantum Computing Basics (similarity: 0.789)
  2. Neural Networks in Healthcare (similarity: 0.165)



## Streamlit Interactive Dashboard
Run this cell to generate a Streamlit app for live interaction with the paper retrieval system.


In [7]:
# Streamlit app info
streamlit_info = """
=== STREAMLIT APP CREATED ===

Interactive Web Dashboard Features:
- Real-time semantic search of papers
- Adjustable result count (1-3 papers)
- Display system metrics and stats
- View paper details in expandable sections
- Beautiful, responsive UI

The app file 'streamlit_app.py' has been created in the Gen_AI directory.
"""

print(streamlit_info)
print("\n📝 INSTRUCTIONS - To launch the live server:")
print("=" * 50)
print("   1. Open a terminal in the Gen_AI folder")
print("   2. Run: streamlit run streamlit_app.py")
print("   3. Opens automatically at http://localhost:8501")
print("=" * 50)
print("\n✓ Streamlit is already installed in your environment")
print("✓ App file: o:\\Programming\\Gen_AI\\streamlit_app.py")



=== STREAMLIT APP CREATED ===

Interactive Web Dashboard Features:
- Real-time semantic search of papers
- Adjustable result count (1-3 papers)
- Display system metrics and stats
- View paper details in expandable sections
- Beautiful, responsive UI

The app file 'streamlit_app.py' has been created in the Gen_AI directory.


📝 INSTRUCTIONS - To launch the live server:
   1. Open a terminal in the Gen_AI folder
   2. Run: streamlit run streamlit_app.py
   3. Opens automatically at http://localhost:8501

✓ Streamlit is already installed in your environment
✓ App file: o:\Programming\Gen_AI\streamlit_app.py


## Summary

You now have a complete AI-powered paper retrieval system with:

1. **Notebook Backend** - Jupyter implementation with embedding model and semantic search
2. **Streamlit Frontend** - Interactive web dashboard at `localhost:8501`
3. **Features**:
   - Real-time search across papers
   - Similarity scoring for each result
   - Adjustable result count
   - Paper browsing interface
   - System metrics display

### How to Use:
1. Enter a search query in the search box
2. See papers ranked by relevance with similarity percentages
3. Adjust the number of results in the sidebar
4. Browse individual papers when no search is active

### Files Created:
- `streamlit_app.py` - Standalone Streamlit application
